# Step 12c — Synthesis injection targets (cap N, no min-balance)

> Select up to **N = 50** instances of ONE `(smell, polarity)` bucket from the unbalanced ground truth, as **injection targets** for synthetic-smell generation (the `smell-synthesizer` repo). Same `(priority, file_key)` sort + cap as Step 12b, but **without** the min-balancing — so a low-positive smell (e.g. `UnsafeDeployOnFork`, only 5 positives) still yields the full 50 negatives to inject into.

**Why:** smells with too few real positives can't be evaluated/balanced. We synthesize positives by injecting the smell into confirmed-clean (absent) files; this step picks WHICH files, deterministically and consistently with the balanced ground truth.

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/12_ground_truth_unbalanced.json` | JSON | Step 12 |
| `data/10_ground_truth_sample.json` | JSON | Step 10 (priority lookup) |
| `data/10b_ground_truth_resample.json` | JSON | Step 10b (priority lookup) |
| `data/12b_final_ground_truth.json` | JSON | Step 12b (subset integrity check) |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/12c_synthesis_targets_{smell}_{polarity}.json` | JSON (12b_final schema) | `smell-synthesizer` (LLM injection) |
| `data/step12c_report.json` | JSON | audit (input/output hashes, subset check, priority ranks) |


In [1]:
# --- Notebook bootstrap ---
from dataset_utils.notebook_bootstrap import setup

setup()

import sys
from pathlib import Path

# Guarded __file__ hack: make dataset_utils importable when run as a plain script
# without an installed wheel. Harmless in Jupyter (__file__ undefined). Do NOT remove.
try:
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
except NameError:
    pass

from dataset_utils.ground_truth_balancing import SYNTHESIS_REPORT_PATH, N, run_selection

# === EDIT ME: which (smell, polarity) bucket to extract as synthesis targets ===
TARGET_SMELL = "UnsafeDeployOnFork"   # PascalCase smell id (see SMELL_CATALOG)
TARGET_POLARITY = "absent"            # "absent" (negatives, to inject INTO) or "present"
# ==============================================================================

print(f"Step 12c - selecting synthesis targets for ({TARGET_SMELL}, {TARGET_POLARITY})...", flush=True)

# Cap each (smell, polarity) bucket to N using the SAME priority sort as Step 12b,
# but with no min-balancing: we want the full N targets. The selection logic lives
# in dataset_utils.ground_truth_balancing (unit-tested, shared with `balance`);
# this notebook only drives it.
output_path, report = run_selection(TARGET_SMELL, TARGET_POLARITY, n=N)

print("Synthesis targets written.")
print(f"  Output : {output_path}")
print(f"  Report : {SYNTHESIS_REPORT_PATH}")
print("-" * 48)
print(f"  Available in bucket : {report['available_in_bucket']}")
print(f"  Selected (cap N={report['n_per_polarity_cap']})  : {report['n_selected']}")
print(f"  Capped              : {report['capped']}")
sub = report['subset_of_final_ground_truth']
print(
    f"  Subset of 12b_final : ok={sub['ok']} "
    f"({len(sub['final_gt_target_keys'])} committed in 12b, "
    f"{len(sub['missing_from_selection'])} missing)"
)

Step 12c - selecting synthesis targets for (UnsafeDeployOnFork, absent)...


Synthesis targets written.
  Output : data/12c_synthesis_targets_UnsafeDeployOnFork_absent.json
  Report : data/step12c_report.json
------------------------------------------------
  Available in bucket : 57
  Selected (cap N=50)  : 50
  Capped              : True
  Subset of 12b_final : ok=True (5 committed in 12b, 0 missing)
